In [3]:
import requests 
import minsearch
import numpy as np
import qdrant_client
import pandas as pd

In [4]:
# import the documents

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')
ground_truth

[{'question': 'When does the course begin?',
  'course': 'data-engineering-zoomcamp',
  'document': 'c02e79ef'},
 {'question': 'How can I get the course schedule?',
  'course': 'data-engineering-zoomcamp',
  'document': 'c02e79ef'},
 {'question': 'What is the link for course registration?',
  'course': 'data-engineering-zoomcamp',
  'document': 'c02e79ef'},
 {'question': 'How can I receive course announcements?',
  'course': 'data-engineering-zoomcamp',
  'document': 'c02e79ef'},
 {'question': 'Where do I join the Slack channel?',
  'course': 'data-engineering-zoomcamp',
  'document': 'c02e79ef'},
 {'question': 'Where can I find the prerequisites for this course?',
  'course': 'data-engineering-zoomcamp',
  'document': '1f6520ca'},
 {'question': 'How do I check the prerequisites for this course?',
  'course': 'data-engineering-zoomcamp',
  'document': '1f6520ca'},
 {'question': 'Where are the course prerequisites listed?',
  'course': 'data-engineering-zoomcamp',
  'document': '1f6520c

In [5]:
# import code for evaluating retrieval

from tqdm.auto import tqdm

def hit_rate(relevance_total): # Calculate the hit rate
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

# Calculate the Mean Reciprocal Rank (MRR)
# MRR is the average of the reciprocal ranks of the first relevant document for each query.
def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

# Define ground truth evaluation function
# This function takes the ground truth data and a search function, and evaluates the search results.
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

# Q1: MinSearch

In [5]:
# Set up minsearch 
import minsearch 
boost = {'question': 1.5, 'section': 0.1}

# Create the index for searching. Specify the fields to be indexed and searchable
index = minsearch.Index(
    text_fields=["question", "text", "section"],  # searchable fields
    keyword_fields=["course"]  # filterable fields
)

# Fit the index with documents
index.fit(documents)

# Define the search function
'''This function will take a ground truth query and return search results. It uses the index to search for the question text, filters by course, and applies boosting. It returns the top 5 results based on the search criteria'''

def minsearch_function(q):
    # q is the ground truth document/record like: {'question': 'Where can I sign up...', 'course': 'machine-learning-zoomcamp', 'document': '0227b872'}
    
    results = index.search(
        query=q['question'],  # The actual question text
        filter_dict={'course': q['course']},  # Filter to the right course
        boost_dict=boost,  # Use the specified boost parameters
        num_results=5  # Return top 5 results
    )
    
    return results

# Testing (one query first)
test_query = ground_truth[0]
print("Test query:", test_query)
print("\nSearch results:")
test_results = minsearch_function(test_query)
for i, result in enumerate(test_results):
    print(f"{i+1}. ID: {result['id']}, Question: {result['question'][:50]}...")
print(f"\nLooking for document ID: {test_query['document']}")

Test query: {'question': 'When does the course begin?', 'course': 'data-engineering-zoomcamp', 'document': 'c02e79ef'}

Search results:
1. ID: c02e79ef, Question: Course - When will the course start?...
2. ID: a482086d, Question: Course - Can I follow the course after it finishes...
3. ID: 7842b56a, Question: Course - Can I still join the course after the sta...
4. ID: 1f6520ca, Question: Course - What are the prerequisites for this cours...
5. ID: 63394d91, Question: Course - What can I do before the course starts?...

Looking for document ID: c02e79ef


In [6]:
# Evaluate metrics

results_eval = evaluate(ground_truth, minsearch_function)
print(f"Hit Rate: {results_eval['hit_rate']:.2f}")
print(f"MRR: {results_eval['mrr']:.2f}")

  0%|          | 0/4627 [00:00<?, ?it/s]

Hit Rate: 0.85
MRR: 0.73


---

# Q2: Vector search for question
Evaluate this seach method. What's **MRR** for it?
- 0.25
- 0.35
- 0.45
- 0.55

### Embeddings

In [7]:
# import packages for vector search
import numpy as np
from minsearch import VectorSearch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

In [8]:
# Step 1: Extract quesiton texts-- create embeddings for the `question` field 

texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

# Step 2: Create embeddings-- convert text to numerical TF-IDF scores and reduce dimensionality with SVD
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3), # remove rare words
    TruncatedSVD(n_components=128, random_state=1) # reduce dimensionality with SVD (128 components)
)

# Fit the pipeline to the texts and transform them into vectors
X = pipeline.fit_transform(texts) # each question = 128 dimensions

In [9]:
# Step 3: Create vector search index 
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [10]:
# Step 4: Create search function for vector search

def vector_search_function(q):
    # q is the ground truth document or record like: {'question': 'Where can I sign up...', 'course': 'machine-learning-zoomcamp', 'document': '0227b872'}
    
    # Transform the question text into a vector using the fitted pipeline
    query_vector = pipeline.transform([q['question']])
    
    # Search the vector index for the top 5 results
    results = vindex.search(query_vector, filter_dict={'course': q['course']}, num_results=5)
    
    return results

In [11]:
### TESTING FUNCTION 
test_query = ground_truth[0]
print("Test query:", test_query)

test_results = vector_search_function(test_query)
print(f"\nVector search results:")
for i, result in enumerate(test_results):
    print(f"{i+1}. ID: {result['id']}, Question: {result['question'][:50]}...")
    
print(f"\nLooking for document ID: {test_query['document']}")

Test query: {'question': 'When does the course begin?', 'course': 'data-engineering-zoomcamp', 'document': 'c02e79ef'}

Vector search results:
1. ID: c02e79ef, Question: Course - When will the course start?...
2. ID: 7842b56a, Question: Course - Can I still join the course after the sta...
3. ID: 138b55c7, Question: Edit Course Profile....
4. ID: 0bbf41ec, Question: Course - I have registered for the Data Engineerin...
5. ID: a482086d, Question: Course - Can I follow the course after it finishes...

Looking for document ID: c02e79ef


In [12]:
# Step 5: Evaluate MRR (Mean Reciprocal Rank) for vector search

results = evaluate(ground_truth, vector_search_function)
print(f"Vector Search MRR: {results['mrr']:.2f}")
print(f"Vector Search Hit Rate: {results['hit_rate']:.2f}")

  0%|          | 0/4627 [00:00<?, ?it/s]

Vector Search MRR: 0.36
Vector Search Hit Rate: 0.48


- **ANSWER**: 0.36. So the VECTOR SEARCH isn't actually better so far than `minsearch` in this case yet. Could be bc only used on `question`, not on `answer` (or the *text*). 

---

## Q3: Vector Search for both `question` and `text` (answer) 

In [13]:
# Q3: Extract question + answer text (instead of just questions)

texts = []

for doc in documents:
    # Combine question and text (answer) with a space
    t = doc['question'] + ' ' + doc['text']
    texts.append(t)

# Step 2: Create embeddings for question + answer text
# Use the SAME pipeline as Q2 (min_df=3, n_components=128)
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Fit and transform the enhanced texts
X_enhanced = pipeline.fit_transform(texts)
print(f"Enhanced embeddings shape: {X_enhanced.shape}") 

Enhanced embeddings shape: (948, 128)


In [14]:
# Step 3: Create enhanced vector search index with question + answer text
vindex_enhanced = VectorSearch(keyword_fields={'course'})
vindex_enhanced.fit(X_enhanced, documents)

In [15]:
# Check what ground truth looks like
print("Ground truth sample:", ground_truth[0])

Ground truth sample: {'question': 'When does the course begin?', 'course': 'data-engineering-zoomcamp', 'document': 'c02e79ef'}


In [16]:
# Step 4: Define the enhanced vector search function for question + answer text

def vector_search_function_enhanced(q):
    # q is the ground truth document (note that in this example, it only has the `question`, not the `answer` or text)
    
    # Transform the question text into a vector using the fitted pipeline
    query_text = q['question']  # the ground truth only contains `question`-- don't have the `answer` or text for the query
    
    query_vector = pipeline.transform([query_text])
    
    # Evaluate results using the enhanced vector index
    results = vindex_enhanced.search(
        query_vector=query_vector[0],
        filter_dict={'course': q['course']},
        num_results=5
    )
    
    return results

In [17]:
# Step 5: Run full evaluation for enhanced vector search

print("\nRunning full enhanced vector evaluation...")
results = evaluate(ground_truth, vector_search_function_enhanced)
print(f"Enhanced Vector Search Hit Rate: {results['hit_rate']:.2f}")
print(f"Enhanced Vector Search MRR: {results['mrr']:.2f}")


Running full enhanced vector evaluation...


  0%|          | 0/4627 [00:00<?, ?it/s]

Enhanced Vector Search Hit Rate: 0.82
Enhanced Vector Search MRR: 0.67


---

# Q4: Qdrant

- Basically doing the same thing as Q3 but using `qdrant` and **professional embeddings**. 
- *Question*: What is the **MRR**?

In [6]:
# Step 1: Qdrant set-up (vector database)
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from sentence_transformers import SentenceTransformer
from qdrant_client import models

# Initialize Qdrant client (in-memory for homework)
client = QdrantClient(":memory:")  # Local, temporary database

# Initialize Jina embedding model
model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en")
print("Jina model loaded successfully!")

Some weights of BertModel were not initialized from the model checkpoint at jinaai/jina-embeddings-v2-small-en and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.intermediate.dense.bias', 'encoder.layer.1.intermediate.dense.weight', 'encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.1.output.LayerNorm.weight', 'encoder.layer.1.output.dense.bias', 'encoder.layer.1.output.dense.weight', 'encoder.layer.2.intermediate.dense.bias', 'encoder.layer.2.intermediate.dense.weight', 'encoder.layer.2.output.LayerNorm.bias', 'encoder.layer.2.output.LayerNorm.weight', 'encoder.layer.2.output.dense.bias', 'encoder.layer.2.output.dense.weight', 'encoder.layer.3.intermediate.dense.bias', 'encoder.layer.3.intermediate.den

Jina model loaded successfully!


In [ ]:
# Step 2. Create embeddings using Jina model
texts = []
for doc in documents:
    text = doc['question'] + ' ' + doc['text']
    texts.append(text)

print(f"Creating Jina embeddings for {len(texts)} documents...")

embeddings = model.encode(texts, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")  # Should be (948, 512) - Jina uses 512 dimensions

Creating Jina embeddings for 948 documents...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Embeddings shape: (948, 512)


In [ ]:
# Step 3: Create collection and UPLOAD embeddings to Qdrant!
from qdrant_client.models import Distance, VectorParams, PointStruct
collection_name = "course-questions"

try: # Check if collection exists and delete it if so
    if client.get_collection(collection_name):
        client.delete_collection(collection_name)
    print(f"✅ Deleted existing collection '{collection_name}'")
except Exception as e:
    print(f"Collection didn't exist or couldn't delete: {e}")

# Now create a fresh collection
try:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embeddings.shape[1],  # Should be 512 for Jina
            distance=Distance.COSINE
        )
    )
    print(f"✅ Created fresh collection '{collection_name}'")
except Exception as e:
    print(f"❌ Error creating collection: {e}")

# Create points using PointStruct (proper Qdrant format)
embedding = model.encode([doc['question'] + ' ' + doc['text']])[0]
points = []
for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
    point = PointStruct(
        id=i, # PointStruct requires a unique ID, using integer index here
        vector=embedding.tolist(),  # Convert numpy array to list
        payload={
            "question": doc['question'],
            "text": doc['text'],
            "course": doc['course'],
            "section": doc['section'],
            "id": doc['id']  # Keep original document ID in payload
        }
    )
    points.append(point)

print(f"Prepared {len(points)} PointStruct objects")

# Upload all points to Qdrant in smaller batches (more reliable)
batch_size = 100
total_uploaded = 0

try:
    for i in range(0, len(points), batch_size):
        batch = points[i:i+batch_size]
        
        client.upsert(
            collection_name=collection_name,
            points=batch
        )
        total_uploaded += len(batch)
        print(f"Uploaded batch {i//batch_size + 1}: {len(batch)} points (total: {total_uploaded})")
    
    print(f"✅ Successfully uploaded all {total_uploaded} points!")
    
    # Verify upload worked
    collection_stats = client.count(collection_name)
    print(f"Final count in collection: {collection_stats}")
    
except Exception as e:
    print(f"❌ Error uploading points: {e}")
    print(f"Error type: {type(e)}")

Collection didn't exist or couldn't delete: Collection course-questions not found
✅ Created fresh collection 'course-questions'
Prepared 948 PointStruct objects
Uploaded batch 1: 100 points (total: 100)
Uploaded batch 2: 100 points (total: 200)
Uploaded batch 3: 100 points (total: 300)
Uploaded batch 4: 100 points (total: 400)
Uploaded batch 5: 100 points (total: 500)
Uploaded batch 6: 100 points (total: 600)
Uploaded batch 7: 100 points (total: 700)
Uploaded batch 8: 100 points (total: 800)
Uploaded batch 9: 100 points (total: 900)
Uploaded batch 10: 48 points (total: 948)
✅ Successfully uploaded all 948 points!
Final count in collection: count=948


### 👉🏻 Step 4: Create search function for Qdrant

In [ ]:
# Step 4: create Qdrant Search Function 

def qdrant_search_function(q):
    """
    Input: q = {"question": "...", "course": "..."}
    Output: List of 5 documents (as dictionaries) <-- top 5 results based on the question and course filter
    """
    
    # 1. Encode the QUERY question (ground truth only has `question` field)
    # query_embedding = model.encode([q['question']])[0]
    # or experiment with 
    query_embedding = model.encode([q['question'] + ' '])[0]
    
    # 2. Search in Qdrant with course filter
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=q['course'])
                )
            ]
        ),
        limit=5,
        with_payload=True 
    )
    
    # 3. Convert results to expected format
    results = []
    for hit in search_result.points:
        results.append(hit.payload)
    
    return results

In [ ]:
# # Check what collections exist
# collections = client.get_collections()
# print("Available collections:", collections)

# # Check collection stats
# stats = client.count("course-questions")
# print(f"Documents in 'course-questions': {stats}")

Available collections: collections=[CollectionDescription(name='course-questions')]
Documents in 'course-questions': count=948


In [ ]:
# # Test with a known good query
# test_query = {"question": "Course start date?", "course": "data-engineering-zoomcamp"}

# print("🔍 Testing qdrant_search_function...")
# try:
#     results = qdrant_search_function(test_query)
#     print(f"✅ Function returned {len(results)} results")
    
#     if results:
#         for i, result in enumerate(results):
#             print(f"  {i+1}: {result.get('id')} - {result.get('question', 'NO QUESTION')[:50]}...")
#     else:
#         print("❌ No results returned!")
        
# except Exception as e:
#     print(f"❌ Function crashed: {type(e).__name__}: {e}")

🔍 Testing qdrant_search_function...
✅ Function returned 5 results
  1: 1f6520ca - Course - What are the prerequisites for this cours...
  2: 10515af5 - Are we still using the NYC Trip data for January 2...
  3: 690ba010 - GCS Bucket - error when writing data from web to G...
  4: 9336ce2c - Compressed file ended before the end-of-stream mar...
  5: 3e0114ad - Can I use Airflow instead for my final project?...


In [15]:
# Step 5: run full evaluation for Qdrant + Jina

print("Running full Qdrant + Jina evaluation...")
qdrant_results = evaluate(ground_truth, qdrant_search_function)
print(f"Q4 Final Results:")
print(f"Qdrant + Jina MRR: {qdrant_results['mrr']:.2f}")
print(f"Qdrant + Jina Hit Rate: {qdrant_results['hit_rate']:.2f}")

# Compare with your previous results
print(f"\nComparison:")
print(f"Q1 (Text Search):      Hit Rate = 0.84")
print(f"Q2 (Basic Vector):     MRR = 0.35") 
print(f"Q3 (Enhanced Vector):  Hitrate = 0.82")
print(f"Q3 (Enhanced Vector):  MRR = 0.62")
print(f"Q4 (Qdrant + Jina):    Hit Rate = {qdrant_results['hit_rate']:.2f}")
print(f"Q4 (Qdrant + Jina):    MRR = {qdrant_results['mrr']:.2f}")

Running full Qdrant + Jina evaluation...


  0%|          | 0/4627 [00:00<?, ?it/s]

Q4 Final Results:
Qdrant + Jina MRR: 0.11
Qdrant + Jina Hit Rate: 0.16

Comparison:
Q1 (Text Search):      Hit Rate = 0.84
Q2 (Basic Vector):     MRR = 0.35
Q3 (Enhanced Vector):  Hitrate = 0.82
Q3 (Enhanced Vector):  MRR = 0.62
Q4 (Qdrant + Jina):    Hit Rate = 0.16
Q4 (Qdrant + Jina):    MRR = 0.11


In [155]:
# Quick sanity check - does Qdrant even have the right data?
print("🔍 Quick Qdrant debug:")
print(f"Collection count: {client.count('course-questions')}")

# Test search without any filter
no_filter_result = client.query_points(
    collection_name="course-questions",
    query=model.encode(["test"])[0].tolist(),
    limit=3,
    with_payload=True
)
print(f"No-filter search returns: {len(no_filter_result.points)} results")
if no_filter_result.points:
    print(f"Sample result: {no_filter_result.points[0].payload.get('question', 'NO QUESTION')[:50]}")

🔍 Quick Qdrant debug:
Collection count: count=948
No-filter search returns: 3 results
Sample result: Computing the hash for project review


In [152]:
# Test your embedding similarity manually
if found_doc:
    query_text = "Course start date?"
    doc_text = found_doc['question'] + ' ' + found_doc['text']
    
    query_emb = model.encode([query_text])[0]
    doc_emb = model.encode([doc_text])[0]
    
    # Calculate cosine similarity manually
    import numpy as np
    similarity = np.dot(query_emb, doc_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb))
    print(f"Manual similarity: {similarity:.4f}")

Manual similarity: 0.6033


In [156]:
# Test 1: Search WITHOUT course filter
def qdrant_search_no_filter(q):
    query_embedding = model.encode([q['question']])[0]
    
    search_result = client.query_points(
        collection_name="course-questions",
        query=query_embedding.tolist(),
        # NO FILTER AT ALL
        limit=5,
        with_payload=True 
    )
    
    results = []
    for hit in search_result.points:
        results.append(hit.payload)
    return results

# Test this version
test_query = {"question": "Course start date?", "course": "data-engineering-zoomcamp"}
no_filter_results = qdrant_search_no_filter(test_query)

print("No-filter search results:")
for i, result in enumerate(no_filter_results):
    print(f"  {i}: {result.get('id')} - {result.get('question')[:50]}... (Course: {result.get('course')})")

No-filter search results:
  0: 6a417bfe - How to get started with Week 10?... (Course: machine-learning-zoomcamp)
  1: ff40f83b - How to get started with Week 8?... (Course: machine-learning-zoomcamp)
  2: 3ee083ab - How to get started with Week 9?... (Course: machine-learning-zoomcamp)
  3: cbf13b19 - Where is the FAQ for Prefect questions?... (Course: mlops-zoomcamp)
  4: 1d644223 - Will I get a certificate if I missed the midterm p... (Course: machine-learning-zoomcamp)


In [157]:
# Check the ground truth size vs document collection
print(f"Ground truth test cases: {len(ground_truth)}")
print(f"Documents in collection: 948")

# Check if ground truth document IDs exist in your document collection
doc_ids = set(doc['id'] for doc in documents)
gt_doc_ids = set(gt['document'] for gt in ground_truth)

print(f"Unique document IDs in collection: {len(doc_ids)}")
print(f"Unique target IDs in ground truth: {len(gt_doc_ids)}")
print(f"Ground truth IDs missing from collection: {len(gt_doc_ids - doc_ids)}")

# Show some missing IDs
missing_ids = gt_doc_ids - doc_ids
if missing_ids:
    print(f"Sample missing IDs: {list(missing_ids)[:5]}")

Ground truth test cases: 4627
Documents in collection: 948
Unique document IDs in collection: 947
Unique target IDs in ground truth: 947
Ground truth IDs missing from collection: 0


In [158]:
# Check if course filtering works at all
test_query = {"question": "Course start date?", "course": "data-engineering-zoomcamp"}

# Test with broken filter (your current function)
results_with_filter = qdrant_search_function(test_query)
print(f"With course filter: {len(results_with_filter)} results")

# Test without filter
results_no_filter = qdrant_search_no_filter(test_query)  
print(f"Without course filter: {len(results_no_filter)} results")

# Compare the results
print("\nWith filter - courses found:")
for r in results_with_filter:
    print(f"  {r.get('course')}")
    
print("\nWithout filter - courses found:")
for r in results_no_filter:
    print(f"  {r.get('course')}")

With course filter: 5 results
Without course filter: 5 results

With filter - courses found:
  data-engineering-zoomcamp
  data-engineering-zoomcamp
  data-engineering-zoomcamp
  data-engineering-zoomcamp
  data-engineering-zoomcamp

Without filter - courses found:
  machine-learning-zoomcamp
  machine-learning-zoomcamp
  machine-learning-zoomcamp
  mlops-zoomcamp
  machine-learning-zoomcamp


In [159]:
# Check how many times each document is tested
from collections import Counter
doc_counts = Counter(gt['document'] for gt in ground_truth)

print(f"Most tested document appears {max(doc_counts.values())} times")
print(f"Average tests per document: {len(ground_truth) / len(doc_counts):.1f}")

# Show some examples
print("\nSample document test frequencies:")
for doc_id, count in list(doc_counts.most_common(5)):
    print(f"  Document {doc_id}: {count} test queries")

Most tested document appears 5 times
Average tests per document: 4.9

Sample document test frequencies:
  Document c02e79ef: 5 test queries
  Document 1f6520ca: 5 test queries
  Document 7842b56a: 5 test queries
  Document 0bbf41ec: 5 test queries
  Document 63394d91: 5 test queries


In [160]:
# Testing without filters 
# Final test: no-filter evaluation
print("Testing no-filter version...")
qdrant_results_no_filter = evaluate(ground_truth, qdrant_search_no_filter)
print(f"No-filter results:")
print(f"Hit Rate: {qdrant_results_no_filter['hit_rate']:.3f}")
print(f"MRR: {qdrant_results_no_filter['mrr']:.3f}")

Testing no-filter version...


  0%|          | 0/4627 [00:00<?, ?it/s]

No-filter results:
Hit Rate: 0.105
MRR: 0.066


In [ ]:
# Test without course filter
def qdrant_search_no_filter(q):
    query_embedding = model.encode([q['question']])[0]
    
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        # NO FILTER - search everything
        limit=5,
        with_payload=True 
    )
    
    results = []
    for hit in search_result.points:
        results.append(hit.payload)
    
    return results

# Test the same query without filter
test_results_no_filter = qdrant_search_no_filter(test_query)
print("Results WITHOUT course filter:")
for i, result in enumerate(test_results_no_filter):
    print(f"  {i}: {result.get('id')} - {result.get('question')[:50]}... (Course: {result.get('course')})")

In [ ]:
print(f"Test query course: '{test_query['course']}'")
print(f"Expected doc course: '{found_doc['course']}'")
print(f"Match: {test_query['course'] == found_doc['course']}")

### Troubleshooting

In [ ]:
# Testing both approaches 
# Quick comparison test
print("=== COMPARISON TEST ===")

# Test question + text approach (your current)
rank_enhanced = find_target_rank(test_query, max_results=50)
print(f"Question + Text approach: Target rank = {rank_enhanced}")

# Test question-only approach  
def find_target_rank_q_only(q, max_results=50):
    query_embedding = model.encode([q['question']])
    search_result = client.query_points(
        collection_name=collection_name_q_only,
        query=query_embedding[0].tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="course", 
                    match=models.MatchValue(value=q['course'])
                )
            ]
        ),
        limit=max_results,
        with_payload=True
    )
    
    target_id = q['document']
    for i, hit in enumerate(search_result.points):
        if hit.payload['id'] == target_id:
            return i + 1
    return -1

rank_q_only = find_target_rank_q_only(test_query, max_results=50)
print(f"Question-only approach: Target rank = {rank_q_only}")

## More troubleshooting

In [161]:
# Check specific problem: heavily tested documents
doc_counts = Counter(gt['document'] for gt in ground_truth)

print(f"Most tested document appears {max(doc_counts.values())} times")
print(f"Average tests per document: {len(ground_truth) / len(doc_counts):.1f}")

# Show the most problematic documents
print("\nMost heavily tested documents:")
for doc_id, count in doc_counts.most_common(10):
    print(f"  Document {doc_id}: {count} test queries")
    
# Find this document in your collection
most_tested_id = doc_counts.most_common(1)[0][0]
most_tested_doc = None
for doc in documents:
    if doc['id'] == most_tested_id:
        most_tested_doc = doc
        break

if most_tested_doc:
    print(f"\nMost tested document:")
    print(f"  ID: {most_tested_doc['id']}")
    print(f"  Course: {most_tested_doc['course']}")
    print(f"  Question: {most_tested_doc['question']}")

Most tested document appears 5 times
Average tests per document: 4.9

Most heavily tested documents:
  Document c02e79ef: 5 test queries
  Document 1f6520ca: 5 test queries
  Document 7842b56a: 5 test queries
  Document 0bbf41ec: 5 test queries
  Document 63394d91: 5 test queries
  Document 2ed9b986: 5 test queries
  Document 93e2c8ed: 5 test queries
  Document a482086d: 5 test queries
  Document eb56ae98: 5 test queries
  Document 4292531b: 5 test queries

Most tested document:
  ID: c02e79ef
  Course: data-engineering-zoomcamp
  Question: Course - When will the course start?


In [162]:
# Test the most problematic document
most_tested_queries = [gt for gt in ground_truth if gt['document'] == most_tested_id]
print(f"\nSample queries for most tested document:")
for i, gt in enumerate(most_tested_queries[:3]):
    print(f"  Query {i+1}: {gt['question']}")
    
    # Test where this document ranks
    results = qdrant_search_function(gt)
    for j, result in enumerate(results):
        if result['id'] == gt['document']:
            print(f"    Target found at position {j+1}")
            break
    else:
        print(f"    Target NOT found in top 5!")


Sample queries for most tested document:
  Query 1: When does the course begin?
    Target NOT found in top 5!
  Query 2: How can I get the course schedule?
    Target NOT found in top 5!
  Query 3: What is the link for course registration?
    Target NOT found in top 5!


In [167]:
# SOLUTION: Use question-only embeddings (like your successful Q2/Q3)

# Step 1: Create question-only embeddings with Jina
print("Creating question-only Jina embeddings...")
question_texts = [doc['question'] for doc in documents]
question_embeddings = model.encode(question_texts, show_progress_bar=True)

# Step 2: Delete and recreate collection with question-only embeddings
client.delete_collection("course-questions")
client.create_collection(
    collection_name="course-questions",
    vectors_config=VectorParams(size=512, distance=Distance.COSINE)
)

# Step 3: Upload question-only embeddings
points = []
for i, (doc, embedding) in enumerate(zip(documents, question_embeddings)):
    point = PointStruct(
        id=i,
        vector=embedding.tolist(),
        payload={
            "question": doc['question'],
            "text": doc['text'],
            "course": doc['course'],
            "section": doc['section'],
            "id": doc['id']
        }
    )
    points.append(point)

# Upload in batches
for i in range(0, len(points), 100):
    batch = points[i:i+100]
    client.upsert(collection_name="course-questions", points=batch)

print("✅ Question-only re-indexing complete!")

# Step 4: Use simple search function (question vs question)
def qdrant_search_simple(q):
    query_embedding = model.encode([q['question']])[0]  # Just the question
    
    search_result = client.query_points(
        collection_name="course-questions",
        query=query_embedding.tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=q['course'])
                )
            ]
        ),
        limit=5,
        with_payload=True 
    )
    
    results = []
    for hit in search_result.points:
        results.append(hit.payload)
    return results

# Test it
test_results = qdrant_search_simple(test_query)
print("Simple approach results:")
for i, result in enumerate(test_results):
    print(f"  {i+1}: {result.get('id')} - {result.get('question')[:50]}...")

Creating question-only Jina embeddings...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Question-only re-indexing complete!
Simple approach results:
  1: c02e79ef - Course - When will the course start?...
  2: 63394d91 - Course - What can I do before the course starts?...
  3: a482086d - Course - Can I follow the course after it finishes...
  4: 2f19301f - How can we contribute to the course?...
  5: 7842b56a - Course - Can I still join the course after the sta...


In [168]:
# Run the full evaluation with your working function
print("🔄 Running final Q4 evaluation...")
qdrant_results_simple = evaluate(ground_truth, qdrant_search_simple)

print(f"\n🎯 Q4 FINAL Results:")
print(f"Hit Rate: {qdrant_results_simple['hit_rate']:.3f}")
print(f"MRR: {qdrant_results_simple['mrr']:.3f}")

print(f"\n📊 Complete Module 3 Results:")
print(f"Q1 (MinSearch Text):      Hit Rate = 0.84")
print(f"Q2 (TF-IDF Vector):       MRR = 0.35")
print(f"Q3 (Enhanced TF-IDF):     Hit Rate = 0.82, MRR = 0.62")
print(f"Q4 (Jina Embeddings):     Hit Rate = {qdrant_results_simple['hit_rate']:.3f}, MRR = {qdrant_results_simple['mrr']:.3f}")

🔄 Running final Q4 evaluation...


  0%|          | 0/4627 [00:00<?, ?it/s]


🎯 Q4 FINAL Results:
Hit Rate: 0.302
MRR: 0.223

📊 Complete Module 3 Results:
Q1 (MinSearch Text):      Hit Rate = 0.84
Q2 (TF-IDF Vector):       MRR = 0.35
Q3 (Enhanced TF-IDF):     Hit Rate = 0.82, MRR = 0.62
Q4 (Jina Embeddings):     Hit Rate = 0.302, MRR = 0.223


- **Answer**: MRR is 0.11 🤔. Not sure...

# Q5: Cosine similarity
- **Question**: What's the average cosine?

In [169]:
# Define cosine similarity function
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

In [170]:
# Using results from gpt-4o-mini evaluations: 
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

In [171]:
# Check what's in this data
print("Q5 Data Preview:")
print(f"Shape: {df_results.shape}")
print(f"Columns: {list(df_results.columns)}")
print(f"\nFirst few rows:")
print(df_results.head(2))

Q5 Data Preview:
Shape: (1830, 5)
Columns: ['answer_llm', 'answer_orig', 'document', 'question', 'course']

First few rows:
                                          answer_llm  \
0  You can sign up for the course by visiting the...   
1  You can sign up using the link provided in the...   

                                         answer_orig  document  \
0  Machine Learning Zoomcamp FAQ\nThe purpose of ...  0227b872   
1  Machine Learning Zoomcamp FAQ\nThe purpose of ...  0227b872   

                              question                     course  
0  Where can I sign up for the course?  machine-learning-zoomcamp  
1   Can you provide a link to sign up?  machine-learning-zoomcamp  


In [ ]:
# Step 1: Create the pipeline (same as Q2/Q3 approach)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Fit the pipeline on ALL text data (LLM answers + original answers + questions)
all_text_data = (df_results.answer_llm + ' ' + 
                 df_results.answer_orig + ' ' + 
                 df_results.question)

print("Fitting pipeline on all text data...")
pipeline.fit(all_text_data)
print("✅ Pipeline fitted!")

Fitting pipeline on all text data...
✅ Pipeline fitted!


In [176]:
# Step 2: Define cosine similarity function 

def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

In [177]:
# Step 3: Calcuate similarities 

similarities = []

for idx, row in df_results.iterrows():
    # Transform both answers to vectors
    llm_answer_vector = pipeline.transform([row['answer_llm']])[0]
    orig_answer_vector = pipeline.transform([row['answer_orig']])[0]
    
    # Calculate cosine similarity
    similarity = cosine(llm_answer_vector, orig_answer_vector)
    similarities.append(similarity)
    
    # Show progress for first few
    if idx < 3:
        print(f"Row {idx}: Similarity = {similarity:.4f}")

print(f"✅ Calculated {len(similarities)} similarities")

Row 0: Similarity = 0.4635
Row 1: Similarity = 0.7816
Row 2: Similarity = 0.8892
✅ Calculated 1830 similarities


In [178]:
# Step 4: Calculate average cosine similarity

average_cosine = np.mean(similarities)
print(f"\n🎯 Q5 Answer:")
print(f"Average Cosine Similarity: {average_cosine:.4f}")

# Show some statistics
print(f"\nSimilarity Statistics:")
print(f"Min: {np.min(similarities):.4f}")
print(f"Max: {np.max(similarities):.4f}")  
print(f"Median: {np.median(similarities):.4f}")
print(f"Std: {np.std(similarities):.4f}")


🎯 Q5 Answer:
Average Cosine Similarity: 0.8416

Similarity Statistics:
Min: 0.0791
Max: 0.9965
Median: 0.9058
Std: 0.1737


## Q6- ROUGE

In [180]:
# Q6: ROUGE Metrics Calculation
import rouge 

# Step 1: Install rouge-score if needed (might need to pip install)
try:
    from rouge_score import rouge_scorer
    print("✅ ROUGE library available")
except ImportError:
    print("❌ Need to install: pip install rouge-score")
    
# Step 2: Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

# Step 3: Calculate ROUGE for each answer pair
print("Calculating ROUGE scores...")

rouge_scores = []

for idx, row in df_results.iterrows():
    score = scorer.score(row['answer_orig'], row['answer_llm']) # F1 score from ROUGE
    rouge1_f1 = score['rouge1'].fmeasure
    rouge_scores.append(rouge1_f1)
    
    # Show progress for first few
    if idx < 3:
        print(f"Row {idx}: ROUGE-1 F1 = {rouge1_f1:.4f}")

print(f"✅ Calculated {len(rouge_scores)} ROUGE scores")

# Step 4: Get average ROUGE score
average_rouge = np.mean(rouge_scores)
print(f"\n🎯 Q6 Answer:")
print(f"Average ROUGE-1 F1: {average_rouge:.4f}")

✅ ROUGE library available
Calculating ROUGE scores...
Row 0: ROUGE-1 F1 = 0.1707
Row 1: ROUGE-1 F1 = 0.2619
Row 2: ROUGE-1 F1 = 0.5600
✅ Calculated 1830 ROUGE scores

🎯 Q6 Answer:
Average ROUGE-1 F1: 0.4510


- **Answer**: 0.45 (average ROUGE-1 F1)